
# 🧠 NumPy Lab: ndarrays & Vectorization (Beginner → Intermediate)

**Duration:** ~60 minutes  
**Goal:** Get comfortable with NumPy `ndarray` basics (shape, dtype, memory) and learn how vectorization speeds up ML-style computations.

---

## What you'll do
1. Generate a small **synthetic dataset** (student exam example).
2. Inspect array **shape, dtype, size, memory, and strides**.
3. Practice **vectorized** math vs. Python loops.
4. Build a **simple vectorized predictor** (logistic-style) and check accuracy.

> Tip: Run cells **top to bottom**.


## 0️⃣ Setup

In [ ]:

import numpy as np
import time

# Reproducibility
rng = np.random.default_rng(123)

np.__version__



## 1️⃣ Create a Simple Synthetic Dataset

We'll simulate two features per student:
- `study_hours` (roughly normal around 5 hours)
- `attendance_rate` (between 50% and 100%)

We will generate labels (pass/fail) using a simple rule with a bit of noise.


In [ ]:

# Number of samples
n_samples = 500

# Features
study_hours = rng.normal(loc=5, scale=2, size=n_samples)          # ~N(5, 2)
attendance_rate = rng.uniform(low=0.5, high=1.0, size=n_samples)  # [0.5, 1.0]

# Combine into (n_samples, 2) matrix
X = np.column_stack([study_hours, attendance_rate])

# Simple scoring rule with noise
linear_combo = 0.6 * X[:, 0] + 0.4 * X[:, 1] * 10  # scale attendance
noise = rng.normal(0, 0.5, size=n_samples)
scores_true = linear_combo + noise

# Binary labels: pass if score > 4.5
y = (scores_true > 4.5).astype(np.int64)

X.shape, y.shape, y[:10]



## 2️⃣ Understanding `ndarray`s

Inspect shape, dtype, itemsize, and total bytes. Then look at array flags and strides.


In [ ]:

print("Shape:", X.shape)
print("Data type:", X.dtype)
print("Item size (bytes):", X.itemsize)
print("Total size (bytes):", X.nbytes)


In [ ]:

X.flags['C_CONTIGUOUS'], X.flags['F_CONTIGUOUS'], X.strides



### 2.1 Views vs Copies

- `.view()` shares memory with the original array.  
- `.copy()` duplicates data.

We'll demonstrate, then **rebuild** `X` to avoid leaving it modified.


In [ ]:

# Show view vs copy behavior
X_view = X.view()
X_copy = X.copy()

print("View shares memory with X:", X_view.base is X)
print("Copy shares memory with X:", X_copy.base is X)

# Modify the view and see the change in X
original = X[0, 0]
X_view[0, 0] = 999
print("After modifying view, X[0,0] =", X[0, 0])

# Restore: rebuild X cleanly to keep downstream cells sane
# (Re-run the generation logic to avoid accidental side-effects)
study_hours = rng.normal(loc=5, scale=2, size=n_samples)
attendance_rate = rng.uniform(low=0.5, high=1.0, size=n_samples)
X = np.column_stack([study_hours, attendance_rate])

linear_combo = 0.6 * X[:, 0] + 0.4 * X[:, 1] * 10
noise = rng.normal(0, 0.5, size=n_samples)
scores_true = linear_combo + noise
y = (scores_true > 4.5).astype(np.int64)

print("Restored X[0,0]:", X[0, 0])



## 3️⃣ Vectorization vs Loops

We'll compare computing the **average study hours** with:
- a Python loop
- a vectorized NumPy operation


In [ ]:

# Loop version
total = 0.0
for h in study_hours:
    total += h
avg_loop = total / len(study_hours)
avg_loop


In [ ]:

# Vectorized version
avg_vec = np.mean(study_hours)
avg_vec


In [ ]:

# Simple timing comparison (small scale; illustrative)
t0 = time.perf_counter()
_ = sum(study_hours) / len(study_hours)
t1 = time.perf_counter()
loop_ms = (t1 - t0) * 1e3

t0 = time.perf_counter()
_ = np.mean(study_hours)
t1 = time.perf_counter()
vec_ms = (t1 - t0) * 1e3

loop_ms, vec_ms, f"Vectorized is about {loop_ms / vec_ms:.1f}× faster (small sample)"



## 4️⃣ Simple Vectorized Predictor (Logistic-Style)

We'll **manually set weights** and compute pass probabilities with a **sigmoid**.  
This is not a full training loop—just a clean example of vectorized inference.


In [ ]:

# Manually chosen weights and bias
weights = np.array([0.8, 1.5])
bias = -4.0

# Linear scores via matrix-vector multiply
scores = X @ weights + bias

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

probs = sigmoid(scores)

probs[:10], probs.shape


In [ ]:

# Convert probabilities to predictions
y_pred = (probs >= 0.5).astype(np.int64)

# Accuracy against our generated labels
accuracy = (y_pred == y).mean()
accuracy



## 5️⃣ Memory & Layout Quick Demos

Contiguity matters for certain low-level operations and sometimes for performance.


In [ ]:

subset = X[:, ::2]  # take every other column
print("subset shape:", subset.shape)
print("C_CONTIGUOUS:", subset.flags['C_CONTIGUOUS'])
print("F_CONTIGUOUS:", subset.flags['F_CONTIGUOUS'])
print("strides:", subset.strides)


In [ ]:

# Reinterpret memory as bytes works best for contiguous arrays
try:
    raw = np.ascontiguousarray(X).view(np.uint8)
    print("Viewing contiguous X as bytes succeeded. Shape:", raw.shape)
except ValueError as e:
    print("Error:", e)



## 6️⃣ Wrap-Up

**Discuss:**
- Why is NumPy faster than loops?
- When is `.view()` appropriate vs `.copy()`?
- How did broadcasting / `@` simplify the prediction step?

**Stretch:**
1. Add a third feature (e.g., `sleep_hours`) and update predictions.
2. Standardize features: `X = (X - X.mean(axis=0)) / X.std(axis=0)`
3. Write a reusable `accuracy(X, y, w, b)` function.
4. Increase `n_samples` and compare loop vs vectorized timing again.
